In [2]:
import os
import pandas as pd
import numpy as np


# =========================
# 1. Route setting
# =========================
# 读数据
players = pd.read_parquet("E:/players.parquet")
games = pd.read_parquet("E:/games.parquet")
events = pd.read_parquet("E:/events.parquet")

# 看基本形状
print("players shape:", players.shape)
print("games shape:", games.shape)
print("events shape:", events.shape)

# 看列名
print("\nplayers columns:")
print(players.columns.tolist())

print("\ngames columns:")
print(games.columns.tolist())

print("\nevents columns:")
print(events.columns.tolist())

# =========================
# 3. Merge game-level outcome info
# =========================
players2 = players.merge(
    games[["game_id", "winner_team", "last_day", "n_players", "end_reason"]],
    on="game_id",
    how="left"
)

print("\nMerged players preview:")
print(players2.head())


# =========================
# 4. Build team / win indicator
# =========================
# Standardize team label based on role
players2["team"] = players2["role"].apply(
    lambda x: "Werewolves" if str(x).strip().lower() == "werewolf" else "Villagers"
)

players2["win"] = (players2["team"] == players2["winner_team"]).astype(int)


# =========================
# 5. Role-level summary
# =========================
# NOTE:
# eliminated_during_day is NOT 0/1; it indicates day index of elimination
# eliminated_during_phase is the correct field for Day / Night / None

role_summary = (
    players2.groupby("role")
    .agg(
        count=("player_id", "count"),
        n_survived=("alive_end", "sum"),
        survive_rate=("alive_end", "mean"),
        day_elim_rate=("eliminated_during_phase", lambda x: (x == "Day").mean()),
        night_elim_rate=("eliminated_during_phase", lambda x: (x == "Night").mean()),
        win_rate=("win", "mean")
    )
    .reset_index()
)

print("\nRole summary:")
print(role_summary)


# =========================
# 6. Optional: role survival detail
# =========================
role_survival = (
    players2.groupby("role")
    .agg(
        n_players=("player_id", "count"),
        n_survived=("alive_end", "sum"),
        survive_rate=("alive_end", "mean")
    )
    .reset_index()
)

print("\nRole survival summary:")
print(role_survival)


# =========================
# 7. Optional: role outcome detail
# =========================
role_outcome_summary = (
    players2.groupby("role")
    .agg(
        win_rate=("win", "mean"),
        avg_last_day=("last_day", "mean"),
        avg_n_players=("n_players", "mean")
    )
    .reset_index()
)

print("\nRole outcome summary:")
print(role_outcome_summary)




players shape: (11472, 7)
games shape: (1435, 6)
events shape: (407262, 17)

players columns:
['game_id', 'player_id', 'role', 'model_name', 'alive_end', 'eliminated_during_day', 'eliminated_during_phase']

games columns:
['game_id', 'filename', 'winner_team', 'last_day', 'n_players', 'end_reason']

events columns:
['game_id', 'filename', 'outer_idx', 'inner_idx', 'data_type', 'event_name', 'day', 'phase', 'detailed_phase', 'source', 'public', 'visible_in_ui', 'created_at', 'actor_id', 'target_id', 'reasoning', 'description']

Merged players preview:
    game_id player_id      role               model_name  alive_end  \
0  74788902    Jordan  Werewolf  Grok 4.1 Fast Reasoning      False   
1  74788902     Jamie  Villager                   Grok 4       True   
2  74788902      Alex      Seer                   Grok 4      False   
3  74788902     Casey  Villager   Gemini 3 Flash Preview       True   
4  74788902     Quinn  Werewolf          Claude Opus 4.6      False   

   eliminated_du